# Chạy LBP hàng loạt cho 10 ảnh (Phiên bản Vector hóa tối ưu bằng NumPy)
Notebook này sử dụng thuật toán LBP đã được vector hóa (vectorization) bằng các phép toán ma trận của thư viện NumPy. Việc xử lý được tối ưu hóa để chạy song song trên CPU, giảm thời gian xử lý ảnh Full HD từ vài phút xuống **dưới 1 giây** mỗi ảnh, trong khi kết quả toán học thu được hoàn toàn trùng khớp 100% với phiên bản chạy vòng lặp truyền thống.

In [1]:
import os
import time
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
print("Done")

Done


In [2]:
def noi_suy_song_tuyen_vectorized(anh, y_coords, x_coords):
    """
    Nội suy song tuyến đồng thời cho toàn bộ mảng tọa độ thực x_coords, y_coords.
    anh: ma trận ảnh xám 2D (float64)
    x_coords, y_coords: ma trận 2D chứa tọa độ thực của các điểm lân cận cần nội suy
    """
    rows, cols = anh.shape
    
    x0 = np.floor(x_coords).astype(np.int32)
    y0 = np.floor(y_coords).astype(np.int32)
    x1 = x0 + 1
    y1 = y0 + 1
    
    x0 = np.clip(x0, 0, cols - 1)
    x1 = np.clip(x1, 0, cols - 1)
    y0 = np.clip(y0, 0, rows - 1)
    y1 = np.clip(y1, 0, rows - 1)
    
    dx = x_coords - x0
    dy = y_coords - y0
    
    f00 = anh[y0, x0]
    f01 = anh[y0, x1]
    f10 = anh[y1, x0]
    f11 = anh[y1, x1]
    
    return (1 - dy) * (1 - dx) * f00 + (1 - dy) * dx * f01 + dy * (1 - dx) * f10 + dy * dx * f11


In [3]:
def tinh_lbp_vectorized(anh_xam, P, R):
    """
    Tính toán LBP sử dụng vector hóa (vectorization) tăng tốc bằng NumPy.
    """
    rows, cols = anh_xam.shape
    anh_lbp = np.zeros((rows, cols), dtype=np.uint8)
    bien = int(np.ceil(R))
    
    r_idx, c_idx = np.meshgrid(
        np.arange(bien, rows - bien),
        np.arange(bien, cols - bien),
        indexing='ij'
    )
    
    gc = anh_xam[r_idx, c_idx]
    bits = []
    
    for p in range(P):
        goc = 2 * np.pi * p / P
        xp = c_idx + R * np.cos(goc)
        yp = r_idx - R * np.sin(goc)
        
        gp = noi_suy_song_tuyen_vectorized(anh_xam, yp, xp)
        bit_p = (gp >= gc).astype(np.uint8)
        bits.append(bit_p)
        
    so_nhom = P // 8
    ma_tran_nhom = []
    
    for g in range(so_nhom):
        nhom_bits = bits[g * 8 : (g + 1) * 8]
        val = np.zeros_like(r_idx, dtype=np.uint32)
        for i, b in enumerate(nhom_bits):
            val += b.astype(np.uint32) * (2 ** i)
        ma_tran_nhom.append(val)
        
    if so_nhom == 1:
        res = ma_tran_nhom[0]
    else:
        res = np.maximum.reduce(ma_tran_nhom)
        
    anh_lbp[r_idx, c_idx] = res.astype(np.uint8)
    return anh_lbp


In [4]:
def tinh_lbp_toan_anh(anh_xam, P, R):
    """
    Áp dụng LBP lên toàn bộ ảnh xám sử dụng giải pháp vector hóa tối ưu.
    """
    return tinh_lbp_vectorized(anh_xam, P, R)


In [5]:
# Cấu hình đường dẫn và danh sách ảnh
input_dir = os.path.join("anh_xlas", "anh_xlas")
output_dir = "ket_qua_lbp"
os.makedirs(output_dir, exist_ok=True)

cau_hinh = [
    (8, 1, 'P=8, R=1'),
    (8, 2, 'P=8, R=2'),
    (16, 2, 'P=16, R=2'),
    (16, 3, 'P=16, R=3'),
    (24, 3, 'P=24, R=3')
]

mau_sac = ['#FF6B6B', '#FFA07A', '#4ECDC4', '#45B7D1', '#96CEB4']
ban_do_mau = ['hot', 'cool', 'inferno', 'plasma', 'viridis']

supported_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
all_files = sorted(os.listdir(input_dir))
image_files = [f for f in all_files if f.lower().endswith(supported_extensions)]
print(f"Tìm thấy {len(image_files)} ảnh trong thư mục '{input_dir}':", image_files)

Tìm thấy 10 ảnh trong thư mục 'anh_xlas/anh_xlas': ['image_01.jpg', 'image_02.jpg', 'image_03.jpg', 'image_04.jpg', 'image_05.jpg', 'image_06.jpg', 'image_07.jpg', 'image_08.jpg', 'image_09.jpg', 'image_10.jpg']


In [ ]:
# Chạy thuật toán và lưu kết quả (hiển thị log tiến độ)
t_start_all = time.time()

for idx, img_name in enumerate(image_files, 1):
    img_path = os.path.join(input_dir, img_name)
    print(f"\n[{idx}/{len(image_files)}] Đang xử lý: {img_name}...")
    
    try:
        # 1. Đọc ảnh và chuyển sang ảnh xám
        img_pil = Image.open(img_path).convert('RGB')
        img_rgb = np.array(img_pil, dtype=np.float64)
        R_ch, G_ch, B_ch = img_rgb[:,:,0], img_rgb[:,:,1], img_rgb[:,:,2]
        img_gray = 0.299*R_ch + 0.587*G_ch + 0.114*B_ch
        
        base_name = os.path.splitext(img_name)[0]
        
        # 2. Xử lý LBP cho cả 5 cấu hình
        ket_qua_lbp = {}
        t_start_img = time.time()
        for P, R, nhan in cau_hinh:
            t0 = time.time()
            anh_lbp = tinh_lbp_toan_anh(img_gray, P, R)
            ket_qua_lbp[nhan] = anh_lbp
            print(f"  - {nhan}: {time.time() - t0:.2f}s")
            
        print(f"  => Hoàn thành {img_name} sau {time.time() - t_start_img:.2f} giây.")
        
        # 3. Vẽ và lưu ảnh so sánh kết quả LBP
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        fig.suptitle(f"Kết quả LBP - {img_name}", fontsize=16, fontweight='bold')
        ax_list = axes.ravel()
        
        ax_list[0].imshow(img_gray, cmap='gray', vmin=0, vmax=255)
        ax_list[0].set_title('Ảnh xám gốc', fontsize=12, fontweight='bold')
        ax_list[0].axis('off')
        
        for i, ((nhan, anh_lbp), bm) in enumerate(zip(ket_qua_lbp.items(), ban_do_mau)):
            ax = ax_list[i+1]
            ax.imshow(anh_lbp, cmap=bm, vmin=0, vmax=255)
            ax.set_title(nhan, fontsize=12, fontweight='bold')
            ax.axis('off')
            
        plt.tight_layout()
        out_img_path = os.path.join(output_dir, f"{base_name}_lbp_ket_qua.png")
        plt.savefig(out_img_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        # 4. Vẽ và lưu histogram
        fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
        fig.suptitle(f"Histogram phân bố giá trị LBP - {img_name}", fontsize=14, fontweight='bold')
        
        for ax, ((nhan, anh_lbp), mau) in zip(axes, zip(ket_qua_lbp.items(), mau_sac)):
            pixels_hop_le = anh_lbp[anh_lbp > 0].ravel()
            ax.hist(pixels_hop_le, bins=32, range=(0,255), color=mau,
                    edgecolor='white', linewidth=0.5, alpha=0.9)
            ax.set_title(nhan, fontsize=11, fontweight='bold', color=mau)
            ax.set_xlabel('Giá trị LBP', fontsize=9)
            ax.set_ylabel('Số pixel', fontsize=9)
            ax.set_facecolor('#F8F9FA')
            for spine in ax.spines.values():
                spine.set_edgecolor('#DDDDDD')
        
        plt.tight_layout()
        out_hist_path = os.path.join(output_dir, f"{base_name}_lbp_histogram.png")
        plt.savefig(out_hist_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"  -> Đã lưu ảnh kết quả tại: {out_img_path}")
        print(f"  -> Đã lưu histogram tại: {out_hist_path}")
        
    except Exception as e:
        print(f"Lỗi khi xử lý ảnh {img_name}: {str(e)}")

print(f"\n Hoàn thành toàn bộ {len(image_files)} ảnh trong {time.time() - t_start_all:.2f} giây!")


[1/10] Đang xử lý: image_01.jpg...
  - P=8, R=1: 1.69s
  - P=8, R=2: 1.53s
  - P=16, R=2: 2.92s
  - P=16, R=3: 2.97s
  - P=24, R=3: 3.70s
  => Hoàn thành image_01.jpg sau 12.81 giây.
  -> Đã lưu ảnh kết quả tại: ket_qua_lbp/image_01_lbp_ket_qua.png
  -> Đã lưu histogram tại: ket_qua_lbp/image_01_lbp_histogram.png

[2/10] Đang xử lý: image_02.jpg...
  - P=8, R=1: 1.35s
  - P=8, R=2: 1.21s
  - P=16, R=2: 2.46s
  - P=16, R=3: 2.47s
  - P=24, R=3: 3.60s
  => Hoàn thành image_02.jpg sau 11.09 giây.
  -> Đã lưu ảnh kết quả tại: ket_qua_lbp/image_02_lbp_ket_qua.png
  -> Đã lưu histogram tại: ket_qua_lbp/image_02_lbp_histogram.png

[3/10] Đang xử lý: image_03.jpg...
  - P=8, R=1: 1.32s
  - P=8, R=2: 1.23s
  - P=16, R=2: 2.45s
  - P=16, R=3: 2.44s
  - P=24, R=3: 3.55s
  => Hoàn thành image_03.jpg sau 10.99 giây.
  -> Đã lưu ảnh kết quả tại: ket_qua_lbp/image_03_lbp_ket_qua.png
  -> Đã lưu histogram tại: ket_qua_lbp/image_03_lbp_histogram.png

[4/10] Đang xử lý: image_04.jpg...
  - P=8, R=1: 1.